In [24]:
import pandas as pd

# Set up

### df loading 


In [25]:
movies = pd.read_csv('/Users/matildedolfato/Desktop/magistrai/soft_eng/project/movies.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [26]:
ratings = pd.read_csv('/Users/matildedolfato/Desktop/magistrai/soft_eng/project/ratings.csv')
ratings = ratings.drop(columns='timestamp')
ratings.head()

,userId,movieId,rating
0,1,17,4.0
1,1,25,1.0
2,1,29,2.0
3,1,30,5.0
4,1,32,5.0


In [27]:
movie_counts = ratings['movieId'].value_counts().reset_index() 
movie_counts.columns = ['movieId', 'count']
movie_counts 

,movieId,count
0,318,102929
1,356,100296
2,296,98409
3,2571,93808
4,593,90330
...,...,...
84427,288825,1
84428,288467,1
84429,287221,1
84430,284087,1


# Reco algorithm

### select top movies

In [28]:
genres = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'Any Genre']
#we can show code of how we obtain this

top_movies = {}
for genre in genres:
    if genre == 'Any Genre':
        mov = movies.merge(movie_counts, on = 'movieId').sort_values(by= 'count', ascending = False) 
        top_movies[genre] = mov.nlargest(20, 'count')
    else: 
        genre_movies = movies[movies['genres'].str.contains(genre, case=False)] #creo un df genre_movies con i movies di quel genere
        genre_movies = genre_movies.merge(movie_counts, on = 'movieId').sort_values(by= 'count', ascending = False)
        top_movies[genre] = genre_movies.nlargest(20, 'count')
#quindi qui noi creiamo un df ogni volta per ogni genere, e mergiamo il count con il df ridotto ogni volta

In [50]:
top_movies['Animation']

,movieId,title,genres,count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,68997
114,4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,54844
6,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,51518
10,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,50442
160,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,46128
124,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,46036
195,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,41463
12,595,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,41342
315,60069,WALL·E (2008),Adventure|Animation|Children|Romance|Sci-Fi,39993
344,68954,Up (2009),Adventure|Animation|Children|Drama,36382


In [ ]:
top_movies['Thriller']

### give movies to rate

In [47]:
def movies_to_rate(genre: str, n = 5):
    return list(top_movies[genre][:n].title)

    


### suggestion algorithm

In [31]:

import numpy as np
from sklearn.metrics.pairwise import euclidean_distances


In [53]:
#here we assume to have n, x (num suggestions), genre (str), ratings (new_rating list)

def suggestion(new_rating: list, genre: str, n = 5, number_of_suggestions = 3):
    new_rating_dict = {}
    for i in range(n):
        new_rating_dict[list(top_movies[genre][:n].movieId)[i]] = new_rating[i] #create ratings list given input by the user

    new_rating_filtered =  {k: v for k, v in new_rating_dict.items() if v != 'Not seen'} #keep only seen ones
    filtered_ratings = ratings[ratings['movieId'].isin(new_rating_filtered.keys())] #filter original df keeping only movies rated by new user 

    pivot_df = filtered_ratings.pivot(index='userId', columns='movieId', values='rating').fillna(2.5) #organise it as original df

    new_user_ratings = pd.DataFrame([new_rating_filtered], index=['new_user']) #create df with ratings by new user
    dissimilarities = euclidean_distances(pivot_df, new_user_ratings) 
    # print('length dissimilrities:',len(dissimilarities)) 
    # num users is 118301
    print(dissimilarities)

    most_similar_user = pivot_df.index[np.argmin(dissimilarities)]
    most_similar_users = pivot_df.index[np.argsort(dissimilarities.flatten())[:20]]


    print(f'Most similar user ID: {most_similar_user}') 

    user_ratings = ratings[ratings['userId'].isin(most_similar_users)]
    #user_ratings = ratings[ratings['userId'] == most_similar_user] #prendo i movies visti dai most similar users
    movies_by_genre = movies[movies['genres'].str.contains(genre, case=False)] #seleziono dal mio dataset originale i movies in base al genere scelto
    user_ratings = user_ratings[user_ratings['movieId'].isin(movies_by_genre.movieId)] #tengo dei movies visti dai miei most similar quelli di quel genere

    user_ratings_filtered = user_ratings.sort_values(by = 'rating', ascending=False) #ordino mio df in base al rating 


    suggested_movies = user_ratings_filtered[~user_ratings_filtered['movieId'].isin(new_rating_filtered.keys())] #prendo quelli che non ha visto
    suggested_movies = suggested_movies.merge(movie_counts).sort_values(by = ['rating', 'count'], ascending=False) #ci aggiungo il count e ranko per rating del mio most similar, e poi per popularità
    # Select the first `n` rows, dropping duplicates based on the 'movieId' column
    final_suggestions_df = suggested_movies.drop_duplicates(subset='movieId').head(number_of_suggestions) #keeps the highest rating, but i don't think it matters now
    final_suggestions_df
    final_suggestions = movies[movies['movieId'].isin(final_suggestions_df.movieId)].title #ritorno il titolo degli x film rated meglio- piu popolari

    # voglio fare un dataset con i piu visti di quel genere togliendo quelli che ha rated e quelli già consigliati FREGANDOCENE DI RATING!! I PIU SEEN!
    # ha senso fare questa dopo la prima, perche nella prima li ordino prima per similarity poi per rating e solo alla fine per count 
    mustsee_suggestions_bygenre=movies_by_genre[~movies_by_genre['movieId'].isin(new_rating_filtered.keys()) 
                                           & ~movies_by_genre['movieId'].isin(final_suggestions_df['movieId'])]
    mustsee_suggestions_df=mustsee_suggestions_bygenre.merge(movie_counts).sort_values(by = ['count'], ascending=False).head(number_of_suggestions)
    mustsee_suggestions=movies[movies['movieId'].isin(mustsee_suggestions_df.movieId)].title

    return final_suggestions, mustsee_suggestions

suggestion([5,5,5,5,5], 'Animation')

[[4.33012702]
 [4.09267639]
 [4.76969601]
 ...
 [4.55521679]
 [5.09901951]
 [1.58113883]]
Most similar user ID: 921


(587     Beauty and the Beast (1991)
 4781          Monsters, Inc. (2001)
 8248        Incredibles, The (2004)
 Name: title, dtype: object,
 5509     Spirited Away (Sen to Chihiro no kamikakushi) ...
 12431                                        WALL·E (2008)
 13363                                            Up (2009)
 Name: title, dtype: object)